# Does minimal representation cost us variants on the real All of Us VDS?

**Run this on the All of Us Researcher Workbench.** It reads only *row*
metadata from the VDS -- no genotypes, no entries -- so it is cheap.

## The question

`aoutools` scores a weights file against the VDS in two steps:

1. `hl.vds.filter_intervals` reads only the loci named in the weights file
   (1bp intervals, built from the **weights** locus, applied to the
   **unsplit** VDS).
2. `hl.vds.split_multi` splits multi-allelic sites, which applies
   `hl.min_rep` to each REF/ALT pair.

The trouble is that `min_rep` can **move a variant's locus**, and the two
steps disagree about where the variant lives.

A joint caller that sees a deletion and a nearby SNP merges them onto one
record. Deleting the `G` at chr1:1002 is anchored one base earlier, so it is
written `chr1:1001 REF=GG ALT=G`; a `G>T` SNP at chr1:1002 becomes `ALT=GT`
on that same record. The VDS row is therefore:

```
chr1:1001   REF=GG   ALT=[G, GT]
```

`min_rep` reduces `GG -> GT` to `G -> T` **at chr1:1002** -- which is exactly
where a GWAS reports that SNP. But the VDS row sits at chr1:1001, so the 1bp
interval built at chr1:1002 never reads it. The variant is silently dropped
before splitting, and never scores.

Worse, `hl.vds.split_multi` is called with the default
`filter_changed_loci=False`, which **raises** on such a variant. It has never
fired in practice -- because the interval filter usually throws those rows
away first. Widening the intervals would make it reachable.

## What this notebook measures

| # | question | what it decides |
|---|---|---|
| 1 | How often does a VDS row have a locus-shifting ALT? | whether this is real at all |
| 2 | How far do loci shift? | the interval pad width, if we fix it |
| 3 | How many rows of a **real PGS file** does it cost? | whether it is worth fixing |
| 4 | Is the `filter_changed_loci` crash reachable? | whether a fix must handle it |

**Decision rule.** If (2) is a negligible fraction of a real scoring file, the
fix is not worth its risk: just make the loss *loud* (drop those alleles
explicitly, count them, warn). If it is material, the fix is to key the join
on the `min_rep`'d variant and pad the intervals -- a bigger change, because
hail will not relocate rows and `hl.vds.split_multi` would have to be replaced.

## Setup

Set `PGS_ID` to a score you actually use. `CONTIG` bounds the cost: the whole
analysis runs on one chromosome and the result extrapolates.

In [ ]:
import collections
import pathlib

import hail as hl
import pandas as pd

from aoutools import get_vds_path, init_hail
from aoutools.prs import download_pgs

PGS_ID = "PGS000746"  # <- a score you actually use
# The notebook's working directory is writable; /home/jupyter may not be.
PGS_DIR = str(pathlib.Path.cwd() / "pgs_downloads")
CONTIG = "chr1"  # one chromosome keeps this cheap; results extrapolate
MAX_PAD = 300  # widest left-pad to consider when sweeping

# A contiguous window, used for the unbiased base rate in Measurement 1.
# Any few Mb of the chromosome will do.
WINDOW = f"{CONTIG}:50000000-60000000"

# init_hail() sets the requester-pays billing project and applies the reference
# genome AFTER hl.init -- passing default_reference to hl.init is deprecated.
# get_vds_path() prefers $WGS_VDS_PATH and falls back to the current All of Us
# release: current Workbench images no longer export that variable, so reading
# it directly (as this cell used to) hands read_vds a None.
init_hail()

VDS_PATH = get_vds_path()
vds = hl.vds.read_vds(VDS_PATH)
print("VDS:", VDS_PATH)

## The mechanism, verified rather than asserted

Before touching the VDS, confirm what `min_rep` does to a handful of
hand-written records. Only shared **leading** bases move a locus, and the
shift is exactly the distance from the deletion's anchor base to the SNP.


In [ ]:
cases = [
    (
        "deletion + SNP one base downstream (a caller emits this)",
        1001,
        ["GG", "G", "GT"],
    ),
    ("deletion + SNP at the anchor base itself", 1001, ["GG", "G", "TG"]),
    (
        "longer deletion, SNP inside the deleted span",
        1001,
        ["GTTTTA", "G", "GTTCTA"],
    ),
    (
        "no shared leading base -- already works today",
        1001,
        ["AGGGC", "A", "GGGGC"],
    ),
]

for label, pos, alleles in cases:
    print(f"{label}\n  chr1:{pos} {alleles}")
    for i in range(1, len(alleles)):
        mr = hl.eval(
            hl.min_rep(hl.locus("chr1", pos), [alleles[0], alleles[i]])
        )
        shift = mr.locus.position - pos
        note = f"SHIFTS +{shift}" if shift else "no shift"
        print(
            f"    alt[{i}]={alleles[i]:<7} -> {mr.locus} {mr.alleles}   {note}"
        )
    print()

## A reusable helper

Explode a VDS's rows to one row per ALT and attach the `min_rep`'d
representation. The physical row never moves -- only the key we *compute*
from it does. That is the whole idea behind the candidate fix, so it is worth
seeing that it works before relying on it.

Star alleles (`*`, a spanning deletion) and symbolic alleles (`<...>`) are
excluded: no GWAS names them, and they would produce junk keys.

In [ ]:
def exploded_alts(variant_rows):
    """One row per (variant, ALT), annotated with its min_rep'd form."""
    ht = variant_rows.annotate(
        alt_idx=hl.range(1, hl.len(variant_rows.alleles))
    )
    ht = ht.explode("alt_idx")
    ht = ht.annotate(alt=ht.alleles[ht.alt_idx])

    # Drop spanning deletions and symbolic alleles.
    ht = ht.filter((ht.alt != "*") & ~ht.alt.startswith("<"))

    ht = ht.annotate(mr=hl.min_rep(ht.locus, [ht.alleles[0], ht.alt]))
    return ht.annotate(
        shift=ht.mr.locus.position - ht.locus.position,
        # The key aoutools joins on: locus + canonical (unordered) allele pair.
        key_locus=ht.mr.locus,
        key_alleles=hl.sorted(ht.mr.alleles),
    )

## Measurement 1 -- the base rate

How often does a VDS row carry an ALT whose `min_rep`'d locus differs from the
row's own locus? Counted over a contiguous window, so it is the unbiased rate:
it does not depend on any weights file.

In [ ]:
window_vds = hl.vds.filter_intervals(
    vds,
    [hl.parse_locus_interval(WINDOW, reference_genome="GRCh38")],
    keep=True,
)
rows = window_vds.variant_data.rows().select()  # row metadata only

alts = exploded_alts(rows).persist()

n_rows = rows.count()
n_multi = rows.filter(hl.len(rows.alleles) > 2).count()
n_alts = alts.count()
n_shifted = alts.filter(alts.shift > 0).count()
n_rows_shifted = alts.filter(alts.shift > 0).select().distinct().count()

print(f"window                       {WINDOW}")
print(f"VDS rows                     {n_rows:,}")
print(
    f"  multi-allelic              {n_multi:,} "
    f"({100 * n_multi / max(n_rows, 1):.2f}%)"
)
print(f"(variant, ALT) pairs         {n_alts:,}")
print(
    f"  ALTs whose locus SHIFTS    {n_shifted:,} "
    f"({100 * n_shifted / max(n_alts, 1):.3f}%)"
)
print(
    f"  rows containing one        {n_rows_shifted:,} "
    f"({100 * n_rows_shifted / max(n_rows, 1):.3f}%)"
)
print()
print("Every one of those shifted ALTs is invisible to aoutools today.")

## Measurement 2 -- how far do they shift?

This sets the interval pad width. The shift can never exceed `len(REF) - 1`,
so the longest deletion in the callset is the hard bound -- but the
*distribution* is what matters, because padding costs VDS read.

In [ ]:
shifted = alts.filter(alts.shift > 0)
dist = collections.Counter(shifted.aggregate(hl.agg.collect(shifted.shift)))

if not dist:
    print("No locus-shifting ALTs in this window at all.")
else:
    total = sum(dist.values())
    cum = 0
    print(f"{'shift':>6} {'count':>9} {'cumulative %':>14}")
    for s in sorted(dist):
        cum += dist[s]
        print(f"{s:>6} {dist[s]:>9,} {100 * cum / total:>13.2f}%")
    print()
    for pct in (0.50, 0.90, 0.99, 1.00):
        need, seen = 0, 0
        for s in sorted(dist):
            seen += dist[s]
            need = s
            if seen / total >= pct:
                break
        print(
            f"  a left-pad of {need:>4} bp would recover "
            f"{100 * pct:.0f}% of them"
        )
    print()
    print(f"  max shift observed: {max(dist)} bp")
    print("  (a pad wider than this buys nothing in this window)")

## Measurement 3 -- what does it cost a *real* PGS file?

This is the number that decides the task. Two counts, on the same weights:

- **reachable today**: the weights row matches a VDS variant *at its own
  locus*, with an ALT whose `min_rep` does **not** move. This is all
  `filter_intervals` + `split_multi` can ever deliver.
- **reachable with the fix**: the weights row matches the `min_rep`'d form of
  *any* ALT, including one whose row physically sits upstream.

The difference is precisely what Task 3 would buy.

In [ ]:
# download_pgs takes keyword arguments only, returns None, and requires
# outdir to already exist. It writes the harmonized scoring file
# (PGS<id>_hmPOS_GRCh38.txt.gz) into that directory.
pathlib.Path(PGS_DIR).mkdir(parents=True, exist_ok=True)
download_pgs(outdir=PGS_DIR, pgs=PGS_ID, build="GRCh38")

found = sorted(pathlib.Path(PGS_DIR).glob(f"{PGS_ID}*.txt.gz"))
if not found:
    raise FileNotFoundError(
        f"no scoring file for {PGS_ID} in {PGS_DIR}; "
        f"found {[p.name for p in pathlib.Path(PGS_DIR).iterdir()]}"
    )
path = found[0]
print("scoring file:", path.name)

w = pd.read_csv(path, sep="\t", comment="#", low_memory=False)
print("columns:", list(w.columns))

# Harmonized files carry hm_chr / hm_pos; fall back to author coordinates.
chr_col = "hm_chr" if "hm_chr" in w else "chr_name"
pos_col = "hm_pos" if "hm_pos" in w else "chr_position"

# other_allele is occasionally absent; the Catalog infers it into
# hm_inferOtherAllele when it can. Without it there is no allele pair to key
# on, so those rows cannot be measured either way.
other = "other_allele"
if other not in w and "hm_inferOtherAllele" in w:
    other = "hm_inferOtherAllele"

w = w[[chr_col, pos_col, "effect_allele", other]].dropna()
w = w.rename(columns={other: "other_allele"})
w[chr_col] = w[chr_col].astype(str).str.replace("chr", "", regex=False)
w = w[w[chr_col] == CONTIG.replace("chr", "")]
w[pos_col] = w[pos_col].astype(int)
print(f"\n{PGS_ID}: {len(w):,} usable rows on {CONTIG}")

In [ ]:
weights = hl.Table.from_pandas(
    w.rename(
        columns={
            chr_col: "chr",
            pos_col: "pos",
            "other_allele": "noneffect_allele",
        }
    )
)
weights = weights.annotate(
    locus=hl.locus("chr" + weights.chr, weights.pos, reference_genome="GRCh38"),
    alleles=hl.sorted([weights.effect_allele, weights.noneffect_allele]),
)
weights = weights.key_by("locus", "alleles").select().distinct().persist()
n_weights = weights.count()

# Read the VDS around every weights locus, padded LEFT by MAX_PAD so that a
# multi-allelic sitting upstream is included. This is the widened interval a
# fix would use; aoutools today uses a 1bp interval and never sees those rows.
wl = weights.key_by("locus").select().distinct()
intervals = wl.annotate(
    interval=hl.interval(
        hl.locus(
            wl.locus.contig,
            hl.max(1, wl.locus.position - MAX_PAD),
            reference_genome="GRCh38",
        ),
        wl.locus,
        includes_end=True,
    )
).key_by("interval")

pgs_vds = hl.vds.filter_intervals(vds, intervals, keep=True)
pgs_alts = exploded_alts(pgs_vds.variant_data.rows().select()).persist()
print(f"weights rows on {CONTIG}: {n_weights:,}")
print(
    "VDS (variant, ALT) pairs read in those padded windows: "
    f"{pgs_alts.count():,}"
)

In [ ]:
def matched(cand):
    """How many weights rows are matched by this candidate allele set?"""
    keyed = cand.key_by(locus=cand.key_locus, alleles=cand.key_alleles)
    keyed = keyed.select().distinct()
    return weights.semi_join(keyed).count()


# What split_multi can deliver: only ALTs that stay put.
today = matched(pgs_alts.filter(pgs_alts.shift == 0))
# What min_rep keying + padded intervals could deliver: all of them.
with_fix = matched(pgs_alts)

recovered = with_fix - today
print(f"{PGS_ID} rows on {CONTIG}      {n_weights:>10,}")
print(
    f"  matched today             {today:>10,} ({100 * today / n_weights:.2f}%)"
)
print(
    f"  matched with the fix      {with_fix:>10,} "
    f"({100 * with_fix / n_weights:.2f}%)"
)
print(
    f"  RECOVERED by the fix      {recovered:>10,} "
    f"({100 * recovered / n_weights:.3f}% of the score)"
)
print()
print("Everything still unmatched after the fix is a different problem:")
print("strand flip, build mismatch, or a variant genuinely absent from AoU.")

### What pad width would those recovered rows actually need?

Padding costs VDS read, so it should be justified, not guessed.

In [ ]:
rec = pgs_alts.filter(pgs_alts.shift > 0)
rec = rec.key_by(locus=rec.key_locus, alleles=rec.key_alleles)
rec = rec.semi_join(weights)  # only the ones that matter to this score

shifts = rec.aggregate(hl.agg.collect(rec.shift))
if not shifts:
    print("The fix would recover nothing on this score; no pad needed.")
else:
    d = collections.Counter(shifts)
    total, cum = sum(d.values()), 0
    print(f"{'pad (bp)':>9} {'recovers':>10} {'cumulative':>12}")
    for s in sorted(d):
        cum += d[s]
        print(f"{s:>9} {d[s]:>10,} {100 * cum / total:>11.1f}%")
    print(f"\n  a {max(d)} bp left-pad recovers all of them")

## Measurement 4 -- is the latent crash reachable?

`hl.vds.split_multi(vds)` uses `filter_changed_loci=False`, which **raises**
on a locus-shifting variant rather than dropping it. It has never fired,
because the 1bp interval filter usually removes those rows first. But if a
weights locus lands exactly on the *physical start* of such a multi-allelic,
the row survives the filter and the run dies.

Widening the intervals pulls many more of these rows in, so any fix must
settle `filter_changed_loci` deliberately.

In [ ]:
risky = pgs_alts.filter(pgs_alts.shift > 0)
risky = risky.key_by("locus").select().distinct()  # PHYSICAL locus

wl_only = weights.key_by("locus").select().distinct()
n_crash_today = wl_only.semi_join(risky).count()

print(
    "weights loci landing on the start of a locus-shifting "
    f"multi-allelic: {n_crash_today:,}"
)
print()
if n_crash_today:
    print("  -> the filter_changed_loci=False crash IS reachable today.")
    print("     That it has not fired is luck, not safety.")
else:
    print("  -> not reachable with 1bp intervals on this score.")
    print("     Padding the intervals WOULD make it reachable, so any")
    print("     fix must set filter_changed_loci deliberately.")

## Verdict

In [ ]:
frac = recovered / n_weights if n_weights else 0.0

print("=" * 68)
print(
    f"Locus-shifting ALTs are {100 * n_shifted / max(n_alts, 1):.3f}% of "
    f"all (variant, ALT) pairs in the window."
)
print(
    f"They cost {PGS_ID} {recovered:,} of {n_weights:,} rows on {CONTIG} "
    f"({100 * frac:.3f}%)."
)
print("=" * 68)
print()

if n_shifted == 0:
    print("VERDICT: the problem does not exist in this callset.")
    print("  AoU never packs a SNP downstream of an indel's anchor base into")
    print("  one record, so every ALT is already minimally represented and")
    print("  min_rep has nothing to relocate. Do not build the fix.")
    print()
    print("  KEEP filter_changed_loci=False (the raising default). With a")
    print("  measured rate of zero it is not a crash risk -- it is a")
    print("  TRIPWIRE. If a future VDS release ever changes representation,")
    print("  it fails loudly instead of silently dropping variants. Setting")
    print("  it True would turn that tripwire into silent data loss.")
elif frac < 0.001:
    print("VERDICT: make the loss LOUD, do not redesign.")
    print(f"  {100 * frac:.3f}% of the score is below the noise floor of a")
    print("  PRS. Replacing hl.vds.split_multi to recover it is not worth the")
    print("  correctness risk. Count the dropped alleles and warn instead.")
else:
    print("VERDICT: worth fixing properly.")
    print(f"  {100 * frac:.3f}% of the score is silently missing, and PRS")
    print("  weights are not uniform -- a dropped variant can carry real")
    print("  signal. Key the join on the min_rep'd variant and left-pad the")
    print("  intervals by the width shown above. Note this means replacing")
    print("  hl.vds.split_multi: hail will not relocate rows, it only raises")
    print("  or drops.")